## Code Review com RAG (Retrieval-Augmented Generation)

Este notebook implementa um pipeline de **Code Review automatizado** utilizando RAG sobre o repositório do LangChain.

### Visão Geral da Arquitetura

O fluxo é dividido em etapas bem definidas:

```
Repositório Git
      │
      ▼
  Carregamento (GenericLoader + LanguageParser)
      │
      ▼
  Chunking (RecursiveCharacterTextSplitter)
      │
      ▼
  Vectorização (OpenAIEmbeddings → Chroma)
      │
      ▼
  Recuperação (MMR Retriever)
      │
      ▼
  Geração (ChatOpenAI + Prompt de Code Review)
```

> **RAG** combina recuperação semântica de contexto relevante com geração de linguagem natural, permitindo que o modelo responda com base em código real do repositório.

### 1. Importações

Importamos as bibliotecas do ecossistema **LangChain** responsáveis por cada camada do pipeline:

| Biblioteca | Responsabilidade |
|---|---|
| `GenericLoader` | Carrega arquivos do sistema de arquivos |
| `LanguageParser` | Faz o parse semântico do código-fonte por linguagem |
| `RecursiveCharacterTextSplitter` | Divide o código em chunks respeitando a estrutura da linguagem |
| `Chroma` | Banco de dados vetorial para armazenar e buscar embeddings |
| `OpenAIEmbeddings` | Transforma os chunks de código em vetores numéricos |
| `ChatOpenAI` | Modelo de linguagem (LLM) responsável pela geração das respostas |
| `ChatPromptTemplate` | Estrutura o prompt enviado ao LLM com contexto e instrução |
| `create_retrieval_chain` | Monta a chain que une retriever + geração de resposta |
| `create_stuff_documents_chain` | Injeta os documentos recuperados diretamente no prompt |

In [4]:
# Loader genérico para carregar documentos de arquivos no sistema
from langchain_community.document_loaders.generic import GenericLoader
# Parser especializado em processar código-fonte de diversas linguagens
from langchain_community.document_loaders.parsers import LanguageParser
# Suporte a linguagens de programação e splitter recursivo para código
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
# Banco de dados vetorial para busca semântica e armazenamento
from langchain_chroma import Chroma
# Gerador de embeddings e interface para o modelo GPT da OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# Template para construção de prompts estruturados para o chat
from langchain_core.prompts import ChatPromptTemplate
# Chain para responder perguntas com base em documentos carregados
from langchain_classic.chains.question_answering import load_qa_chain
# Utilitário para criar uma chain completa de recuperação e resposta
from langchain_classic.chains import create_retrieval_chain
# Chain que combina documentos injetando-os diretamente no prompt
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Interface com o sistema operacional para manipulação de caminhos
import os
# Manipulação de repositórios Git
from git import Repo

### 2. Configuração de Variáveis de Ambiente

Para proteger credenciais sensíveis (como a `OPENAI_API_KEY`), utilizamos o padrão de **variáveis de ambiente** com o arquivo `.env`.

- O arquivo `.env` é carregado via `load_dotenv()`, que injeta as variáveis no ambiente do processo Python.
- Isso evita hardcoding de chaves de API no código-fonte, seguindo as boas práticas de segurança.
- O arquivo `.env` deve estar listado no `.gitignore` para nunca ser versionado.

In [5]:
from dotenv import load_dotenv

load_dotenv()

True

### 3. Clonagem do Repositório Alvo

O primeiro passo do RAG é obter a **base de conhecimento** — neste caso, o código-fonte do repositório que será analisado.

Utilizamos a biblioteca `GitPython` para clonar o repositório do LangChain localmente:

- `repo_path` define o diretório local onde o repositório será armazenado.
- `Repo.clone_from(url, to_path)` realiza o clone via Git, exatamente como `git clone` no terminal.

> Esta célula só precisa ser executada **uma vez**. Em execuções posteriores, o repositório já estará disponível localmente.

In [6]:
repo_path = './repo_review'

In [8]:
repo = Repo.clone_from('https://github.com/langchain-ai/langchain', to_path=repo_path)

### 4. Carregamento dos Documentos (Document Loading)

Com o repositório clonado, utilizamos o `GenericLoader` com o `LanguageParser` para **carregar e parsear os arquivos Python** do repositório.

#### Parâmetros importantes:

| Parâmetro | Valor | Descrição |
|---|---|---|
| `glob` | `'**/*'` | Busca recursiva em todos os subdiretórios |
| `suffixes` | `['.py']` | Filtra apenas arquivos Python |
| `exclude` | `['**/non-utf-8-enconding.py']` | Exclui arquivos com encoding problemático |
| `parser_threshold` | `500` | Arquivos com menos de 500 tokens são tratados como texto simples |

O `LanguageParser` é fundamental para que o código seja dividido de forma **semanticamente coerente** (respeitando funções, classes, etc.), em vez de ser cortado arbitrariamente por caracteres.

In [9]:
loader = GenericLoader.from_filesystem(
    os.path.join(repo_path, 'libs/core/langchain_core'),
    glob='**/*',
    suffixes=['.py'],
    exclude=['**/non-utf-8-enconding.py'],
    parser=LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

documents = loader.load()

In [10]:
len(documents)

641

### 5. Divisão em Chunks (Text Splitting)

Documentos grandes não podem ser enviados inteiros ao LLM devido ao limite de contexto (context window). Por isso, dividimos o código em **chunks menores** usando o `RecursiveCharacterTextSplitter`.

#### Por que `from_language(Language.PYTHON)`?

Este método configura automaticamente os **separadores ideais para Python** (como `\nclass `, `\ndef `, `\n\n`, etc.), garantindo que os cortes respeitem a estrutura do código.

#### Parâmetros de chunking:

| Parâmetro | Valor | Explicação |
|---|---|---|
| `chunk_size` | `2000` | Tamanho máximo de cada chunk (em caracteres) |
| `chunk_overlap` | `200` | Sobreposição entre chunks consecutivos para preservar contexto |

> O `chunk_overlap` é essencial para que trechos de código que atravessam dois chunks não percam seu contexto — por exemplo, uma função que começa no fim de um chunk e continua no próximo.

In [11]:
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=2_000,
    chunk_overlap=200
)

texts = python_splitter.split_documents(documents)

In [12]:
len(texts)

1803

### 6. Vectorização e Criação do Retriever

Esta é a etapa central do RAG: transformamos os chunks de código em **embeddings vetoriais** e os armazenamos no banco de dados vetorial **Chroma**.

#### Como funciona:

1. `OpenAIEmbeddings` converte cada chunk de texto em um vetor numérico de alta dimensão que captura o **significado semântico** do código.
2. `Chroma.from_documents()` persiste esses vetores em memória (ou disco), criando um índice de busca.
3. O `retriever` é configurado para buscar os documentos mais relevantes dada uma query.

#### Por que `search_type='mmr'`?

**MMR (Maximal Marginal Relevance)** equilibra dois objetivos:
- **Relevância**: os chunks retornados devem ser semanticamente próximos da query.
- **Diversidade**: evita retornar chunks muito similares entre si (redundância).

Com `k=8`, o retriever retorna os **8 chunks mais relevantes e diversificados** para compor o contexto da resposta.

> O parâmetro `disallowed_special=()` desabilita erros para tokens especiais, necessário para processar código-fonte que pode conter sequências incomuns.

In [ ]:
db = Chroma.from_documents(texts, OpenAIEmbeddings(disallowed_special=()))

retriever = db.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 8},
)

### 7. Configuração do LLM e do Prompt de Code Review

#### 7.1 Modelo de Linguagem (LLM)

Utilizamos o `gpt-3.5-turbo` via `ChatOpenAI`. O parâmetro `max_tokens=1_000` limita o tamanho da resposta gerada, evitando custos excessivos de API.

#### 7.2 Prompt de Code Review

O `ChatPromptTemplate` estrutura a interação com o LLM em dois papéis:

| Role | Conteúdo |
|---|---|
| `system` | Define a **persona** do modelo como revisor de código Python especializado, com instruções específicas sobre o que analisar |
| `user` | Recebe a **pergunta/solicitação** do usuário sobre o código |

#### Dimensões avaliadas no Code Review:

1. **Bugs/Erros** — lógica incorreta, exceções não tratadas
2. **Performance** — loops ineficientes, operações custosas
3. **Qualidade** — nomes confusos, duplicação, alta complexidade
4. **Python idiomático** — type hints, list comprehensions, context managers

O placeholder `{context}` será preenchido automaticamente com os chunks de código recuperados pelo retriever, e `{input}` com a pergunta do usuário.

In [20]:
llm = ChatOpenAI(model='gpt-3.5-turbo', max_tokens=1_000)

In [21]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            '''Você é um revisor de código Python. Analise o código e aponte:

1. **Bugs/Erros**: lógica incorreta, exceções não tratadas
2. **Performance**: loops ineficientes, operações custosas, uso de estruturas
3. **Qualidade**: nomes confusos, duplicação, complexidade alta
4. **Python idiomático**: use de type hints, list comprehensions, context managers

Seja direto. Format: problema → explicação breve → sugestão.

Contexto:
{context}'''
        ),
        (
            'user',
            '{input}'
        )
    ]
)


### 8. Montagem da Chain RAG

Aqui conectamos todas as peças do pipeline em uma única **chain de recuperação e geração**:

#### `create_stuff_documents_chain(llm, prompt)`
Cria uma chain que recebe documentos e os **"enfia" (stuffs) diretamente no prompt** — daí o nome *stuff*. É a estratégia mais simples: concatena todos os documentos recuperados no campo `{context}` do prompt.

#### `create_retrieval_chain(retriever, document_chain)`
Envolve a `document_chain` com o **retriever**, criando o fluxo completo:

```
query do usuário
      │
      ▼
  retriever.invoke(query)   ← busca os k=8 chunks mais relevantes no Chroma
      │
      ▼
  document_chain.invoke({context: chunks, input: query})   ← gera a resposta
      │
      ▼
  resposta final
```

> Esta abordagem é conhecida como **"Stuff" strategy** no LangChain. Para repositórios muito grandes, alternativas como `MapReduce` ou `Refine` podem ser necessárias para lidar com o limite de contexto do LLM.

In [22]:
document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriever, document_chain)

### 9. Execução do Code Review

Com o pipeline montado, podemos executar queries de code review em linguagem natural.

#### Como o fluxo acontece:

1. A query `'Você pode revisar sugerir melhorias para o código de RunnableBinding'` é passada para a `retrieval_chain`.
2. O **retriever** busca semanticamente no Chroma os 8 chunks de código mais relacionados ao `RunnableBinding`.
3. Esses chunks são injetados no `{context}` do prompt do sistema.
4. O **LLM** analisa o código com base nas instruções de code review e gera a resposta.

#### Resposta retornada:
O objeto `response` contém:
- `response['input']` — a query original
- `response['context']` — os documentos recuperados pelo retriever
- `response['answer']` — a análise gerada pelo LLM

In [24]:
response = retrieval_chain.invoke({
    'input': 'Você pode revisar sugerir melhorias para o código de RunnableBinding'
})

In [25]:
print(response['answer'])

1. **Bugs/Erros**:
   - A classe `RunnableBinding` está definida duas vezes no código, o que pode causar confusão e erros de importação.
  
### Sugestão para Bugs/Erros:
Remova a segunda definição da classe `RunnableBinding`.

2. **Performance**:
   - O código contém trechos extensos de documentação que podem tornar a leitura do código mais difícil, especialmente em funções simples.
   - O uso extensivo de anotações de tipo pode tornar o código mais verboso.
  
### Sugestão de Performance:
Escreva documentação mais concisa e evite anotações de tipo excessivas onde não necessário.

3. **Qualidade**:
   - Os nomes de algumas funções e classes poderiam ser mais descritivos.
   - Alguns trechos de código escondem a lógica real de implementação, o que pode dificultar a compreensão.

### Sugestão de Qualidade:
   - Use nomes mais descritivos para funções e variáveis.
   - Evite esconder a lógica por trás de muitas camadas de abstração.

4. **Python idiomático**:
   - Utilize o método `random